# Региональные ипотечные таблицы: сборка из выгрузки

Ноутбук собирает из сырой выгрузки **«Статистические ряды_регионы.xlsx»** новый файл с листами
**кол-во**, **объем**, **ДДУ** и **Характеристики кредитов** в формате таблицы Дани, с уникальными тикерами
(«Город Москва Льготная ипотека», «Город МоскваВсе программы   Покупка по ДДУ, шт.» и т.д.).

**Как пользоваться:**
1. Меню **Среда выполнения → Выполнить все** (или Ctrl+F9).
2. В шаге 2 нажмите **«Выберите файлы»** и загрузите свежую выгрузку `Статистические ряды_регионы.xlsx`.
3. Подождите 2–3 минуты: готовый файл скачается сам.

В конце выводится проверка: сколько тикеров на каждом листе и есть ли дубли (должно быть 0).

## Шаг 1. Код сборки

Менять ничего не нужно. Тикеры регионов, которые отличаются от названий в выгрузке, заданы в `REGION_TICKERS`.

In [ ]:
"""
Сборка региональных ипотечных таблиц с нуля из сырой выгрузки.

Вход:  «Статистические ряды_регионы.xlsx» (листы 01_02_01, 01_02_03, 01_02_05).
Выход: новый xlsx только с готовыми листами «кол-во», «объем», «ДДУ»,
       «Характеристики кредитов» в формате таблицы Дани, с уникальными тикерами.

Правила (как у Дани):
- кол-во / объем — строки 01_02_01 с размерностью «Всего, шт.» / «Всего, млн руб.»,
  без «Все программы» и «Ипотека в отдельных регионах».
  Тикер = <тикер региона> + название программы («Город Москва Льготная ипотека»).
- ДДУ — строки 01_02_03 «Покупка по ДДУ, шт./млн руб.» по всем программам.
  Ключ = <тикер региона><программа><строка> («Город МоскваВсе программы   Покупка по ДДУ, шт.»).
  Слева служебные формулы Дани в колонках A/B.
- Характеристики кредитов — все строки 01_02_05, ключ как на ДДУ.
- Тикер региона — как у Дани (REGION_TICKERS); для остальных регионов — само название.

Запуск:  python build_regional_tables.py "Статистические ряды_регионы.xlsx" [выходной.xlsx]
Нужен:   pip install openpyxl
"""
import re
import sys
from copy import copy

import openpyxl
from openpyxl.utils import get_column_letter

# Тикеры регионов из таблицы Дани, отличающиеся от названия региона в выгрузке.
# Все прочие регионы (включая ФО, у которых у Дани стоит 0) — тикер = название.
REGION_TICKERS = {
    "Российская Федерация": "Всего РФ",
    "г. Москва": "Город Москва",
    "г. Санкт-Петербург": "Город Санкт-Петербург",
    "г. Севастополь": "Город Севастополь",
    "Республика Татарстан (Татарстан)": "Республика Татарстан",
    "Чувашская Республика - Чувашия": "Чувашская Республика",
    "Республика Адыгея (Адыгея)": "Республика Адыгея",
    "Республика Северная Осетия - Алания": "Республика Северная Осетия",
}

EXCLUDED_PROGRAMS = {"Все программы", "Ипотека в отдельных регионах"}
DDU_RE = re.compile(r"^\s*Покупка по ДДУ,\s*(шт\.|млн руб\.)$")
FIRST_DATA_ROW = 5
HEADER_ROW = 4


def region_ticker(name):
    return REGION_TICKERS.get(name, name)


def region_rows(ws):
    """Строки-заголовки регионов: за ними сразу идёт «Все программы»."""
    last = ws.max_row
    labels = {r: ws.cell(r, 1).value for r in range(FIRST_DATA_ROW, last + 1)}
    return {r for r in range(FIRST_DATA_ROW, last)
            if labels[r + 1] == "Все программы" and labels[r] != "Все программы"}


def col_widths(ws):
    widths = {}
    for dim in ws.column_dimensions.values():
        if dim.width and dim.min and dim.max:
            for c in range(dim.min, dim.max + 1):
                widths[c] = dim.width
    return widths


class SheetWriter:
    """Копирует строки исходного листа в новый, со сдвигом колонок на `shift`."""

    def __init__(self, wb, title, src, shift, top_rows=(1, 2, 3, 4)):
        self.ws = wb.create_sheet(title)
        self.src = src
        self.shift = shift
        self.ncols = src.max_column
        self.row = 0
        for w_col, width in col_widths(src).items():
            self.ws.column_dimensions[get_column_letter(w_col + shift)].width = width
        for r in top_rows:
            self.copy_row(r)

    def copy_row(self, src_row):
        self.row += 1
        for c in range(1, self.ncols + 1):
            s = self.src.cell(src_row, c)
            d = self.ws.cell(self.row, c + self.shift)
            d.value = s.value
            if s.has_style:
                d._style = copy(s._style)
        return self.row

    def set(self, col, value, style_from=None):
        cell = self.ws.cell(self.row, col, value)
        if style_from is not None:
            cell._style = copy(self.ws.cell(self.row, style_from)._style)
        return cell

    def finish(self, freeze, filter_first_col):
        last = get_column_letter(self.ncols + self.shift)
        self.ws.freeze_panes = freeze
        self.ws.auto_filter.ref = f"{filter_first_col}{HEADER_ROW}:{last}{self.row}"


def build_count_volume(wb, src, title, dim_label):
    w = SheetWriter(wb, title, src, shift=1)
    fix_first_col(w, range(1, HEADER_ROW + 1))
    w.ws.column_dimensions["A"].width = col_widths(src).get(1, 24.7)
    w.ws.column_dimensions["B"].width = 24.7
    w.ws.cell(HEADER_ROW, 2, "Тикер")._style = copy(w.ws.cell(HEADER_ROW, 1)._style)
    # служебная строка 3 с номерами колонок для ВПР (как у Дани)
    w.ws.cell(3, 2, 1)
    for c in range(3, src.max_column + 2):
        w.ws.cell(3, c, c)

    regions = region_rows(src)
    prefix = None
    for r in range(FIRST_DATA_ROW, src.max_row + 1):
        label = src.cell(r, 1).value
        if r in regions:
            prefix = region_ticker(label)
            w.copy_row(r)
            fix_first_col(w, [w.row])
            w.set(2, prefix, style_from=1)
            continue
        if src.cell(r, 2).value != dim_label or not isinstance(label, str):
            continue
        if label.strip() in EXCLUDED_PROGRAMS:
            continue
        w.copy_row(r)
        fix_first_col(w, [w.row])
        w.set(2, prefix + label)
    w.finish("D5", "A")
    return w


def fix_first_col(w, rows):
    """copy_row сдвигает всю строку вправо; вернуть колонку подписей в A."""
    for rr in rows:
        b = w.ws.cell(rr, 2)
        a = w.ws.cell(rr, 1)
        a.value = b.value
        a._style = copy(b._style)
        b.value = None


def build_ddu(wb, src):
    w = SheetWriter(wb, "ДДУ", src, shift=3)
    # колонка подписей: из A выгрузки в C, данные с E; D — ключ
    for rr in range(1, w.row + 1):
        move(w.ws, rr, 4, 3)
    w.ws.column_dimensions["A"].width = 8.1
    w.ws.column_dimensions["B"].width = 8.1
    w.ws.column_dimensions["C"].width = col_widths(src).get(1, 47.7)
    w.ws.column_dimensions["D"].width = 46.3
    for c in range(4, src.max_column + 4):
        w.ws.cell(3, c, c - 3)

    regions = region_rows(src)
    prefix = program = None
    for r in range(FIRST_DATA_ROW, src.max_row + 1):
        label = src.cell(r, 1).value
        if not isinstance(label, str):
            continue
        if r in regions:
            prefix, program, key = region_ticker(label), None, region_ticker(label)
        elif src.cell(r, 2).value is None and not label.startswith(" "):
            program, key = label, None
        elif DDU_RE.match(label):
            key = prefix + program + label
        else:
            continue
        w.copy_row(r)
        move(w.ws, w.row, 4, 3)
        w.set(4, key, style_from=3 if key is None else None)
        rr = w.row
        w.ws.cell(rr, 1, 0 if rr == FIRST_DATA_ROW else f"=IF(A{rr-1}=18,0,A{rr-1}+1)")
        w.ws.cell(rr, 2, f"=IF(C{rr}=$C$7,1,IF(C{rr}=$C$8,2,0))")
    w.finish("D5", "C")
    return w


def move(ws, row, from_col, to_col):
    s, d = ws.cell(row, from_col), ws.cell(row, to_col)
    d.value, d._style = s.value, copy(s._style)
    s.value = None


def build_characteristics(wb, src):
    w = SheetWriter(wb, "Характеристики кредитов", src, shift=1)
    fix_first_col(w, range(1, HEADER_ROW + 1))
    w.ws.column_dimensions["A"].width = col_widths(src).get(1, 53.7)
    w.ws.column_dimensions["B"].width = 46.3
    w.ws.cell(HEADER_ROW, 2, "Тикер")._style = copy(w.ws.cell(HEADER_ROW, 1)._style)

    regions = region_rows(src)
    prefix = program = None
    for r in range(FIRST_DATA_ROW, src.max_row + 1):
        label = src.cell(r, 1).value
        if not isinstance(label, str):
            continue
        w.copy_row(r)
        fix_first_col(w, [w.row])
        if r in regions:
            prefix = region_ticker(label)
            w.set(2, prefix, style_from=1)
        elif not label.startswith(" "):
            program = label
        else:
            w.set(2, prefix + program + label)
    w.finish("C5", "A")
    return w


def check_unique(ws, col):
    seen, dups = set(), set()
    for r in range(FIRST_DATA_ROW, ws.max_row + 1):
        v = ws.cell(r, col).value
        if v in (None, ""):
            continue
        (dups if v in seen else seen).add(v)
    return len(seen), dups


def build(raw_path, out_path):
    wb = openpyxl.load_workbook(raw_path)
    for name in ("01_02_01", "01_02_03", "01_02_05"):
        if name not in wb.sheetnames:
            raise SystemExit(f"В выгрузке нет листа {name} — это точно «Статистические ряды_регионы»?")
    raw_names = list(wb.sheetnames)
    print("Собираю «кол-во»...")
    build_count_volume(wb, wb["01_02_01"], "кол-во", "Всего, шт.")
    print("Собираю «объем»...")
    build_count_volume(wb, wb["01_02_01"], "объем", "Всего, млн руб.")
    print("Собираю «ДДУ»...")
    build_ddu(wb, wb["01_02_03"])
    print("Собираю «Характеристики кредитов»...")
    build_characteristics(wb, wb["01_02_05"])
    for name in raw_names:
        del wb[name]
    wb.active = 0

    ok = True
    for name, col in (("кол-во", 2), ("объем", 2), ("ДДУ", 4), ("Характеристики кредитов", 2)):
        n, dups = check_unique(wb[name], col)
        print(f"  {name}: {wb[name].max_row} строк, {n} тикеров, дублей: {len(dups)}")
        if dups:
            ok = False
            print("    дубли:", sorted(dups)[:10])
    wb.save(out_path)
    print(f"Готово: {out_path}" + ("" if ok else "  (ВНИМАНИЕ: есть дубли тикеров)"))
    return ok

## Шаг 2. Загрузить выгрузку и получить готовый файл

In [ ]:
from google.colab import files

uploaded = files.upload()
if not uploaded:
    raise SystemExit("Файл не выбран")
raw_path = next(iter(uploaded))
out_path = "Региональные таблицы (кол-во, объем, ДДУ, характеристики).xlsx"

build(raw_path, out_path)
files.download(out_path)